# Normalization Processing Job

Launches `src/normalize.py` as a SageMaker Processing Job.
Reads the 6 raw source TSVs from S3 and writes normalized versions to `data/normalized/`.

**Run this once.** Output files:
```
data/normalized/train/norm_s1.tsv
data/normalized/train/norm_s2.tsv
data/normalized/train/norm_s3.tsv
data/normalized/test/norm_test_s1.tsv
data/normalized/test/norm_test_s2.tsv
data/normalized/test/norm_test_s3.tsv
```

In [ ]:
import sagemaker
from sagemaker.sklearn import SKLearnProcessor
from sagemaker.processing import ProcessingInput, ProcessingOutput

session = sagemaker.Session()
bucket  = session.default_bucket()
role    = sagemaker.get_execution_role()
PREFIX  = 'entity-resolution-challenge'

print(f'Bucket : s3://{bucket}/{PREFIX}/')
print(f'Role   : {role[:60]}...')

In [ ]:
processor = SKLearnProcessor(
    framework_version='1.2-1',
    role=role,
    instance_type='ml.m5.2xlarge',
    instance_count=1,
    sagemaker_session=session,
)

In [ ]:
processor.run(
    code='src/normalize.py',
    inputs=[
        ProcessingInput(
            source=f's3://{bucket}/{PREFIX}/data/',
            destination='/opt/ml/processing/input',
        )
    ],
    outputs=[
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{bucket}/{PREFIX}/data/normalized/',
        )
    ],
)

print(f'\nDone. Output at: s3://{bucket}/{PREFIX}/data/normalized/')

## Verify output

Check that all 6 files were written and spot-check a few rows.

In [ ]:
import boto3
import pandas as pd

s3 = boto3.client('s3')

expected_keys = [
    f'{PREFIX}/data/normalized/train/norm_s1.tsv',
    f'{PREFIX}/data/normalized/train/norm_s2.tsv',
    f'{PREFIX}/data/normalized/train/norm_s3.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s1.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s2.tsv',
    f'{PREFIX}/data/normalized/test/norm_test_s3.tsv',
]

for key in expected_keys:
    try:
        obj  = s3.head_object(Bucket=bucket, Key=key)
        size = obj['ContentLength'] // 1024
        print(f'  ✓  {key.split("/")[-1]:25s}  {size:>6} KB')
    except Exception:
        print(f'  ✗  MISSING: {key}')

In [ ]:
# Spot-check norm_s1
norm_s1 = pd.read_csv(
    f's3://{bucket}/{PREFIX}/data/normalized/train/norm_s1.tsv',
    sep='\t', dtype=str,
)

print(f'norm_s1: {len(norm_s1):,} rows, {len(norm_s1.columns)} columns')
print(f'Columns: {list(norm_s1.columns)}\n')
norm_s1.head(5)